# 🇮🇩 AksaraLLM — Training Komunitas

[![GitHub](https://img.shields.io/badge/GitHub-AksaraLLM-black?logo=github)](https://github.com/AksaraLLM/aksaraLLM)
[![HuggingFace](https://img.shields.io/badge/🤗-AksaraLLM-yellow)](https://huggingface.co/AksaraLLM)
[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AksaraLLM/aksaraLLM/blob/main/AksaraLLM_Training_Komunitas.ipynb)

> **Model bahasa Indonesia pertama yang dibangun bersama komunitas!**  
> Pipeline lengkap dari nol — **Tokenizer → Pre-training → SFT → DPO** — semua GRATIS di Google Colab!

---

## ⚡ Estimasi Waktu
| Tahap | Waktu |
|-------|-------|
| 🔧 Setup | 3 menit |
| 🔤 Tokenizer | 10 menit |
| 📚 Pre-training | 60 menit |
| 🎓 SFT | 20 menit |
| 🎯 DPO | 20 menit |
| **Total** | **~2 jam** |

## 📋 Yang Dibutuhkan
- Google Account (untuk Drive + Colab gratis)
- HuggingFace Account (untuk upload model — **opsional**)
- Tidak perlu GPU pribadi! T4 Colab sudah cukup ✅

---
### 🚀 Cara Pakai: Klik `Runtime → Run all` dan tunggu!


In [ ]:
#@title ⚙️ Konfigurasi (Ubah jika perlu)
HF_TOKEN = "" #@param {type:"string"}
HF_REPO = "username/AksaraLLM-Mini" #@param {type:"string"}
UPLOAD_TO_HF = False #@param {type:"boolean"}
SKIP_TOKENIZER = False #@param {type:"boolean"}
PRETRAIN_STEPS = 3000 #@param {type:"integer"}
SFT_EPOCHS = 2 #@param {type:"integer"}

print("✅ Konfigurasi disimpan!")
print(f"   Upload ke HF: {UPLOAD_TO_HF}")
print(f"   Pre-training steps: {PRETRAIN_STEPS}")

In [ ]:
#@title 🔧 STEP 0: Setup
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os

# Clone repo
if not os.path.exists('/content/aksaraLLM'):
    subprocess.run(['git', 'clone', 'https://github.com/AksaraLLM/aksaraLLM.git', '/content/aksaraLLM'], check=True)
    print("✅ Repo di-clone!")
else:
    print("✅ Repo sudah ada")

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', 'datasets', 'tokenizers', 'huggingface_hub', '-q'], check=True)

import torch
print(f"\n✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
print(f"   PyTorch: {torch.__version__}")

# Setup dirs
os.makedirs("/content/drive/MyDrive/aksaraLLM-data", exist_ok=True)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("\n🚀 Setup selesai!")

In [ ]:
#@title 🔤 STEP 1: Train Indonesian Tokenizer
# Skip jika sudah ada di Drive
import os
from tokenizers import ByteLevelBPETokenizer

tok_dir = "/content/drive/MyDrive/aksaraLLM-data/aksara-tokenizer-id"

if os.path.exists(f"{tok_dir}/vocab.json") and SKIP_TOKENIZER:
    print("✅ Tokenizer sudah ada di Drive, skip!")
    tok = ByteLevelBPETokenizer(f"{tok_dir}/vocab.json", f"{tok_dir}/merges.txt")
    print(f"   Vocab: {tok.get_vocab_size():,} tokens")
else:
    from datasets import load_dataset
    import time
    t = time.time()
    
    print("📥 Download Wikipedia Indonesia (50k artikel)...")
    wiki = load_dataset("wikimedia/wikipedia", "20231101.id", split="train[:50000]")
    print(f"✅ {len(wiki):,} artikel")
    
    with open("/tmp/wiki_id.txt", "w") as f:
        for art in wiki:
            if len(art["text"]) > 100:
                f.write(art["text"] + "\n")
    
    print("🔤 Training tokenizer 32k vocab...")
    tokenizer = ByteLevelBPETokenizer()
    tokenizer.train(
        files=["/tmp/wiki_id.txt"],
        vocab_size=32_000,
        min_frequency=2,
        special_tokens=["<pad>", "<s>", "</s>", "<unk>",
                        "### Instruksi:", "### Jawaban:", "### Respons:"]
    )
    
    os.makedirs(tok_dir, exist_ok=True)
    tokenizer.save_model(tok_dir)
    
    tok = ByteLevelBPETokenizer(f"{tok_dir}/vocab.json", f"{tok_dir}/merges.txt")
    print(f"\n✅ Tokenizer selesai dalam {time.time()-t:.0f}s!")
    print(f"   Vocab: {tok.get_vocab_size():,} tokens")
    print(f"   Tersimpan di: {tok_dir}")

In [ ]:
#@title 📚 STEP 2: Download & Pre-tokenize Wikipedia
import numpy as np, os, time
from tokenizers import ByteLevelBPETokenizer

tok_dir = "/content/drive/MyDrive/aksaraLLM-data/aksara-tokenizer-id"
tok = ByteLevelBPETokenizer(f"{tok_dir}/vocab.json", f"{tok_dir}/merges.txt")

data_path = "/content/drive/MyDrive/aksaraLLM-data/wiki_tokens.npy"
SEQ_LEN = 256

if os.path.exists(data_path):
    data = np.load(data_path).astype(np.int64)
    print(f"✅ Data dari Drive: {data.shape[0]:,} sequences | {data.shape[0]*SEQ_LEN/1e6:.0f}M tokens")
else:
    from datasets import load_dataset
    print("📥 Download Wikipedia Indonesia (full)...")
    wiki = load_dataset("wikimedia/wikipedia", "20231101.id", split="train")
    print(f"✅ {len(wiki):,} artikel")
    
    print("⚡ Pre-tokenizing...")
    t = time.time()
    all_ids = []
    for i, art in enumerate(wiki):
        ids = tok.encode(art["text"]).ids
        if len(ids) > 50:
            all_ids.extend(ids)
        if (i+1) % 100000 == 0:
            print(f"  {i+1:,} articles | {len(all_ids)/1e6:.0f}M tokens")
    
    total = (len(all_ids) // SEQ_LEN) * SEQ_LEN
    data = np.array(all_ids[:total], dtype=np.int64).reshape(-1, SEQ_LEN)
    np.save(data_path, data.astype(np.uint16))
    print(f"\n✅ Selesai dalam {(time.time()-t)/60:.1f} menit!")
    print(f"   {data.shape[0]:,} sequences | {total/1e6:.0f}M tokens")
    print(f"   Tersimpan di Drive ✅")

In [ ]:
#@title 🔥 STEP 3: Pre-training (60 menit)
import numpy as np, time, math, os, torch
from torch.utils.data import TensorDataset, DataLoader
import sys
sys.path.insert(0, "/content/aksaraLLM")
from aksarallm.model import aksaraLLMModel
from aksarallm.config import aksaraLLMConfig

device = "cuda"
torch.backends.cudnn.benchmark = True
torch.cuda.empty_cache()

# Load data
data = np.load("/content/drive/MyDrive/aksaraLLM-data/wiki_tokens.npy").astype(np.int64)
print(f"✅ Data: {data.shape[0]:,} seqs | {data.shape[0]*256/1e6:.0f}M tokens")

# Model
config = aksaraLLMConfig(
    vocab_size=32_000, n_layers=6, n_heads=6, n_embd=384,
    n_inner=1536, max_seq_len=256, dropout=0.05
)
model = aksaraLLMModel(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"🔥 Model: {n_params/1e6:.1f}M params | VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

# Hyperparams
BATCH_SIZE, GRAD_ACCUM = 32, 4  # effective = 128
EPOCHS = 3
LR_MAX, LR_MIN = 6e-4, 6e-5

loader = DataLoader(
    TensorDataset(torch.from_numpy(data)),
    batch_size=BATCH_SIZE, shuffle=True,
    pin_memory=True, num_workers=2,
    persistent_workers=True, drop_last=True
)

STEPS_PER_EPOCH = len(loader) // GRAD_ACCUM
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS
WARMUP = int(TOTAL_STEPS * 0.05)
print(f"📋 {EPOCHS} epochs | {TOTAL_STEPS:,} steps | eff_batch={BATCH_SIZE*GRAD_ACCUM}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_MAX, betas=(0.9, 0.95), weight_decay=0.1)

def get_lr(s):
    if s < WARMUP: return LR_MAX * s / WARMUP
    p = (s - WARMUP) / (TOTAL_STEPS - WARMUP)
    return LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1 + math.cos(math.pi * p))

scaler = torch.amp.GradScaler("cuda")
step = 0; t0 = time.time(); best_loss = 999
model.train()
print("\n🚀 PRE-TRAINING DIMULAI!")
print("="*60)

for epoch in range(EPOCHS):
    epoch_loss = 0; count = 0
    for i, (batch,) in enumerate(loader):
        x, y = batch[:, :-1].to(device), batch[:, 1:].to(device)
        with torch.amp.autocast("cuda"):
            _, loss = model(x, y)
            loss = loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        epoch_loss += loss.item() * GRAD_ACCUM; count += 1

        if (i + 1) % GRAD_ACCUM == 0:
            lr = get_lr(step)
            for pg in optimizer.param_groups: pg["lr"] = lr
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            step += 1

            if step % 200 == 0:
                elapsed = time.time() - t0
                avg = epoch_loss / count
                spd = step / elapsed
                eta = (TOTAL_STEPS - step) / spd
                vram = torch.cuda.memory_allocated() / 1e9
                print(f"Ep{epoch+1} | {step}/{TOTAL_STEPS} | Loss: {avg:.3f} | "
                      f"LR: {lr:.1e} | {spd:.1f}s/s | ETA: {eta/60:.0f}m | VRAM: {vram:.1f}GB")

            if step % 2000 == 0:
                torch.save({"model_state_dict": model.state_dict(), "config": config.__dict__},
                           f"/content/drive/MyDrive/aksaraLLM-data/pretrain_{step}.pt")
                print(f"💾 Checkpoint saved at step {step}")

    avg = epoch_loss / count
    print(f"\n✅ Epoch {epoch+1} selesai | Loss: {avg:.3f} | {(time.time()-t0)/60:.0f}m")
    if avg < best_loss:
        best_loss = avg
        torch.save({"model_state_dict": model.state_dict(), "config": config.__dict__},
                   "/content/drive/MyDrive/aksaraLLM-data/pretrain_best.pt")
        print(f"🏆 Best model! Loss: {best_loss:.3f}")

total_time = time.time() - t0
print(f"\n{'='*60}")
print(f"🎉 PRE-TRAINING SELESAI!")
print(f"   Loss: {best_loss:.3f} | Waktu: {total_time/60:.0f} menit")
print(f"   Checkpoint: pretrain_best.pt")

In [ ]:
#@title 🎓 STEP 4: SFT - Supervised Fine-Tuning (20 menit)
import json, time, torch, copy
from torch.utils.data import Dataset, DataLoader
from tokenizers import ByteLevelBPETokenizer
from datasets import load_dataset

tok_dir = "/content/drive/MyDrive/aksaraLLM-data/aksara-tokenizer-id"
tok = ByteLevelBPETokenizer(f"{tok_dir}/vocab.json", f"{tok_dir}/merges.txt")
device = "cuda"

# Download Alpaca Indonesia
print("📥 Download Alpaca Indonesia...")
alpaca = load_dataset("cahya/alpaca-id", split="train")
print(f"✅ {len(alpaca):,} samples")

# Identity — model tahu siapa dirinya!
identity_texts = [
    "Di bawah ini adalah instruksi.\n\n### Instruksi:\nSiapa kamu?\n\n### Respons:\nSaya adalah AksaraLLM, asisten AI berbahasa Indonesia yang dibangun oleh komunitas open-source.",
    "Di bawah ini adalah instruksi.\n\n### Instruksi:\nApa itu AksaraLLM?\n\n### Respons:\nAksaraLLM adalah model bahasa Indonesia pertama yang dilatih dari nol oleh komunitas, tersedia gratis untuk semua orang.",
    "Di bawah ini adalah instruksi.\n\n### Instruksi:\nSiapa yang membuat kamu?\n\n### Respons:\nSaya dibuat oleh komunitas AksaraLLM — proyek open-source untuk membangun AI berbahasa Indonesia.",
    "Di bawah ini adalah instruksi.\n\n### Instruksi:\nKamu bisa apa saja?\n\n### Respons:\nSaya bisa menjawab pertanyaan, menjelaskan konsep, menulis teks, dan membantu berbagai tugas dalam bahasa Indonesia.",
] * 80

class SFTDataset(Dataset):
    def __init__(self, hf_data, identities, tokenizer, max_len=256):
        self.samples = []
        texts = [s["text"] for s in hf_data] + identities
        for t in texts:
            ids = tokenizer.encode(t).ids[:max_len]
            if len(ids) > 10:
                self.samples.append(torch.tensor(ids, dtype=torch.long))
        print(f"✅ SFT Dataset: {len(self.samples):,} samples")
    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]

def collate(batch):
    ml = max(x.size(0) for x in batch)
    pad = torch.zeros(len(batch), ml, dtype=torch.long)
    for i, x in enumerate(batch): pad[i, :x.size(0)] = x
    return pad

dataset = SFTDataset(alpaca, identity_texts, tok)
loader = DataLoader(dataset, batch_size=32, shuffle=True,
                    collate_fn=collate, num_workers=2, pin_memory=True)

# Load pretrained
ckpt = torch.load("/content/drive/MyDrive/aksaraLLM-data/pretrain_best.pt",
                  map_location=device, weights_only=False)
cfg = aksaraLLMConfig(**{k:v for k,v in ckpt["config"].items()
                         if k in aksaraLLMConfig.__dataclass_fields__})
model = aksaraLLMModel(cfg).to(device)
model.load_state_dict(ckpt["model_state_dict"], strict=False)
print(f"✅ Model loaded dari pretrain_best.pt")

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda")
t0 = time.time(); best = 999
print("\n🎓 SFT DIMULAI!")

for epoch in range(SFT_EPOCHS):
    model.train(); total = 0
    for step, batch in enumerate(loader):
        x, y = batch[:, :-1].to(device), batch[:, 1:].to(device)
        with torch.amp.autocast("cuda"):
            _, loss = model(x, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        optimizer.zero_grad()
        total += loss.item()
        if (step+1) % 200 == 0:
            avg = total/(step+1)
            elapsed = time.time()-t0
            eta = (len(loader)-step-1)/(((step+1)/elapsed))
            print(f"Ep{epoch+1} | {step+1}/{len(loader)} | Loss: {avg:.4f} | ETA: {eta/60:.0f}m")
    avg = total/len(loader)
    if avg < best:
        best = avg
        torch.save({"model_state_dict": model.state_dict(), "config": cfg.__dict__},
                   "/content/drive/MyDrive/aksaraLLM-data/sft_best.pt")
        print(f"🏆 SFT Epoch {epoch+1} | Loss: {best:.4f} | Saved!")

print(f"\n🎉 SFT SELESAI! | {(time.time()-t0)/60:.0f}m | Loss: {best:.4f}")

In [ ]:
#@title 🎯 STEP 5: DPO - Preference Alignment (20 menit)
import copy, torch, torch.nn.functional as F, time, json
from torch.utils.data import Dataset, DataLoader

# Load SFT model sebagai policy + reference
ckpt = torch.load("/content/drive/MyDrive/aksaraLLM-data/sft_best.pt",
                  map_location=device, weights_only=False)
cfg = aksaraLLMConfig(**{k:v for k,v in ckpt["config"].items()
                         if k in aksaraLLMConfig.__dataclass_fields__})
policy = aksaraLLMModel(cfg).to(device)
policy.load_state_dict(ckpt["model_state_dict"], strict=False)
ref = copy.deepcopy(policy)
for p in ref.parameters(): p.requires_grad_(False)
ref.eval()
print(f"✅ Policy + Reference model siap")

class DPODataset(Dataset):
    def __init__(self, path, tokenizer, max_len=128):
        self.pairs = []
        with open(path) as f:
            for line in f:
                try:
                    d = json.loads(line)
                    c = tokenizer.encode(d["chosen"]).ids[:max_len]
                    r = tokenizer.encode(d["rejected"]).ids[:max_len]
                    if len(c) > 5 and len(r) > 5 and c != r:
                        self.pairs.append((c, r))
                except: pass
        print(f"✅ DPO pairs: {len(self.pairs):,}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i): return self.pairs[i]

def pad_batch(seqs):
    ml = max(len(s) for s in seqs)
    out = torch.zeros(len(seqs), ml, dtype=torch.long)
    for i, s in enumerate(seqs): out[i, :len(s)] = torch.tensor(s)
    return out

def log_probs(m, ids):
    x, y = ids[:, :-1], ids[:, 1:]
    with torch.amp.autocast("cuda"):
        logits, _ = m(x)
    return F.log_softmax(logits, -1).gather(2, y.unsqueeze(2)).squeeze(2).sum(1)

dpo_path = "/content/drive/MyDrive/aksaraLLM-data/hh_rlhf_COMPLETE_160k.jsonl"
if not os.path.exists(dpo_path):
    print("⚠️ File DPO tidak ditemukan, skip DPO...")
    print("   Pakai model SFT saja sebagai final model")
    import shutil
    shutil.copy("/content/drive/MyDrive/aksaraLLM-data/sft_best.pt",
                "/content/drive/MyDrive/aksaraLLM-data/dpo_best.pt")
else:
    dataset = DPODataset(dpo_path, tok)
    loader = DataLoader(dataset, batch_size=16, shuffle=True,
                        collate_fn=lambda b: (pad_batch([x for x,_ in b]),
                                              pad_batch([y for _,y in b])),
                        num_workers=2, pin_memory=True)

    opt = torch.optim.AdamW(policy.parameters(), lr=5e-7, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda")
    best_acc = 0; t0 = time.time()
    print("\n🎯 DPO DIMULAI!")

    for step, (chosen, rejected) in enumerate(loader):
        chosen, rejected = chosen.to(device), rejected.to(device)
        policy.train()
        with torch.no_grad():
            ref_ch, ref_rj = log_probs(ref, chosen), log_probs(ref, rejected)
        pol_ch, pol_rj = log_probs(policy, chosen), log_probs(policy, rejected)
        ratio = 0.1 * ((pol_ch - ref_ch) - (pol_rj - ref_rj))
        loss = -F.logsigmoid(ratio).mean()
        acc = (ratio > 0).float().mean().item()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad()

        if (step+1) % 200 == 0:
            elapsed = time.time()-t0
            eta = (len(loader)-step-1)/((step+1)/elapsed)
            print(f"Step {step+1}/{len(loader)} | Loss: {loss.item():.4f} | Acc: {acc*100:.1f}% | ETA: {eta/60:.0f}m")
        if (step+1) % 500 == 0 and acc > best_acc:
            best_acc = acc
            torch.save({"model_state_dict": policy.state_dict(), "config": cfg.__dict__},
                       "/content/drive/MyDrive/aksaraLLM-data/dpo_best.pt")
            print(f"🏆 Saved! Acc: {best_acc*100:.1f}%")

    # Final save
    if best_acc == 0:
        torch.save({"model_state_dict": policy.state_dict(), "config": cfg.__dict__},
                   "/content/drive/MyDrive/aksaraLLM-data/dpo_best.pt")
    print(f"\n🎉 DPO SELESAI! | {(time.time()-t0)/60:.0f}m | Best Acc: {best_acc*100:.1f}%")

In [ ]:
#@title 💬 STEP 6: Test Model
import torch, sys
from tokenizers import ByteLevelBPETokenizer
sys.path.insert(0, "/content/aksaraLLM")
from aksarallm.model import aksaraLLMModel
from aksarallm.config import aksaraLLMConfig

tok_dir = "/content/drive/MyDrive/aksaraLLM-data/aksara-tokenizer-id"
tok = ByteLevelBPETokenizer(f"{tok_dir}/vocab.json", f"{tok_dir}/merges.txt")
device = "cuda"

ckpt = torch.load("/content/drive/MyDrive/aksaraLLM-data/dpo_best.pt",
                  map_location=device, weights_only=False)
cfg = aksaraLLMConfig(**{k:v for k,v in ckpt["config"].items()
                         if k in aksaraLLMConfig.__dataclass_fields__})
model = aksaraLLMModel(cfg).to(device)
model.load_state_dict(ckpt["model_state_dict"], strict=False)
model.eval()
print(f"✅ Model loaded! {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

def chat(prompt, max_new=100, temp=0.7, top_p=0.9):
    text = f"Di bawah ini adalah instruksi.\n\n### Instruksi:\n{prompt}\n\n### Respons:\n"
    ids = tok.encode(text).ids[-200:]
    ids = torch.tensor([ids], device=device)
    with torch.no_grad():
        for _ in range(max_new):
            logits, _ = model(ids[:, -256:])
            logits = logits[0, -1, :] / temp
            probs = torch.softmax(logits, dim=-1)
            # Top-p sampling
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cumsum = torch.cumsum(sorted_probs, dim=-1)
            mask = cumsum - sorted_probs > top_p
            sorted_probs[mask] = 0
            sorted_probs /= sorted_probs.sum()
            nxt = sorted_idx[torch.multinomial(sorted_probs, 1)]
            ids = torch.cat([ids, nxt.unsqueeze(0).unsqueeze(0)], dim=1)
    output = tok.decode(ids[0].tolist())
    return output.split("### Respons:")[-1].strip()

# Demo!
print("\n" + "="*60)
print("🇮🇩 AKSARALLM DEMO")
print("="*60)

test_questions = [
    "Siapa kamu?",
    "Apa ibu kota Indonesia?",
    "Jelaskan apa itu fotosintesis",
    "Berikan 3 tips belajar yang efektif"
]

for q in test_questions:
    print(f"\n🙋 Q: {q}")
    ans = chat(q)
    print(f"🤖 A: {ans}")
    print("-"*50)

In [ ]:
#@title 🚀 STEP 7: Upload ke HuggingFace (Opsional)
if UPLOAD_TO_HF and HF_TOKEN:
    from huggingface_hub import HfApi, create_repo
    import os

    api = HfApi()
    print(f"📤 Upload ke {HF_REPO}...")
    
    create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)
    
    # Upload model
    api.upload_file(
        path_or_fileobj="/content/drive/MyDrive/aksaraLLM-data/dpo_best.pt",
        path_in_repo="model/dpo_best.pt",
        repo_id=HF_REPO, token=HF_TOKEN
    )
    print("✅ Model uploaded!")
    
    # Upload tokenizer
    tok_dir = "/content/drive/MyDrive/aksaraLLM-data/aksara-tokenizer-id"
    for fname in ["vocab.json", "merges.txt"]:
        api.upload_file(
            path_or_fileobj=f"{tok_dir}/{fname}",
            path_in_repo=f"tokenizer/{fname}",
            repo_id=HF_REPO, token=HF_TOKEN
        )
    print("✅ Tokenizer uploaded!")
    
    print(f"\n🎉 SELESAI! Model tersedia di:")
    print(f"   https://huggingface.co/{HF_REPO}")
else:
    print("⏭️ Upload dilewati.")
    print("   Set UPLOAD_TO_HF=True dan masukkan HF_TOKEN jika ingin upload.")

print("\n" + "="*60)
print("🎊 SELAMAT! AksaraLLM berhasil ditraining!")
print("   Model tersimpan di Google Drive kamu.")
print("   Bergabunglah di Discord: [link discord]")
print("   GitHub: https://github.com/AksaraLLM")
print("="*60)